# Phase02 ? Resume Original3DGS on Kaggle
Private project input only. Enable Internet and one GPU (T4 preferred when available). GPU0 only; two GPUs do not pool VRAM. This notebook does not run until you execute its cells. Save Version with outputs after each completed session.


## 1. Inspect environment and configure paths

In [ ]:
from pathlib import Path
import json, os, shutil, subprocess, sys, platform
INPUT = Path('/kaggle/input/your-private-dataset/kaggle_input')  # folder containing manifest.json
PREVIOUS = None  # optional Path to phase02_results from a saved earlier Kaggle session
WORK = Path('/tmp/phase02-kaggle/project')
OUTPUT = Path('/kaggle/working/phase02_results')
SESSION_TARGET = 30000  # lower absolute target for a shorter session; must exceed checkpoint iteration
print(platform.platform(), sys.version)
subprocess.run(['nvidia-smi'], check=True)
print('Host nvcc:',shutil.which('nvcc'))
print('Free disk GiB:', shutil.disk_usage('/tmp').free / 1024**3)
import torch
print('Host Torch:', torch.__version__, 'CUDA:', torch.version.cuda)
assert torch.cuda.is_available(), 'Select GPU accelerator; CPU training is prohibited'
print('GPU0:',torch.cuda.get_device_name(0), 'VRAM GiB:',torch.cuda.get_device_properties(0).total_memory/1024**3)


## 2. Verify trusted source capsule before extraction
Compare the printed manifest SHA with the Windows preparation report. Only use your own project bundle; checkpoints use Python pickle.

In [ ]:
import hashlib, zipfile
m = json.loads((INPUT/'manifest.json').read_text())
print('Manifest SHA256:',hashlib.sha256((INPUT/'manifest.json').read_bytes()).hexdigest())
archive = INPUT / m['source_capsule']
assert hashlib.sha256(archive.read_bytes()).hexdigest() == m['files'][m['source_capsule']]
assert not WORK.exists(), 'Use a fresh workspace; never overwrite an earlier run'
with zipfile.ZipFile(archive) as z:
    for info in z.infolist():
        target = (WORK/info.filename).resolve()
        assert target.is_relative_to(WORK.resolve()) and '\\' not in info.filename
        assert ((info.external_attr >> 16) & 0o170000) != 0o120000
    z.extractall(WORK)
print('Source base commit:',m['source_base_commit'], 'training code:', m['source_code_hash'])


## 3. Isolated Python3.10 / Torch2.4.1+cu121 / CUDA12.1 build
This explicitly creates a separate pinned environment, leaving Kaggle host packages and driver intact. Setup/solver/build errors stop execution. Linux CUDA binaries are rebuilt; Windows binaries are never transferred.

In [ ]:
with (Path('/kaggle/working')/'phase02_setup.log').open('w') as log:
    result = subprocess.run(['bash',str(WORK/'scripts/kaggle/setup_linux.sh')],stdout=log,stderr=subprocess.STDOUT)
assert result.returncode == 0, 'See phase02_setup.log; do not continue after failed CUDA setup'
PY = WORK/'.tools/b0_env/bin/python'
ENV = os.environ.copy()
ENV['CUDA_VISIBLE_DEVICES'] = '0'
ENV['CUDA_HOME'] = str(WORK/'.tools/b0_env')
def stage(name, extra=()):
    return subprocess.run([str(PY),str(WORK/'scripts/kaggle/runtime.py'),name,'--input',str(INPUT),*map(str,extra)],cwd=WORK,env=ENV,check=True)


## 4. Verify all dataset/checkpoint checksums and prepared geometry

In [ ]:
subprocess.run([str(PY),'-c',"import sys;sys.path.insert(0,'scripts/kaggle');from workflow import verify;from pathlib import Path; m=verify(Path(sys.argv[1]));print(m['checkpoint_validation'])",str(INPUT)],cwd=WORK,env=ENV,check=True)


## 5. Derive checkpoint identity and perform actual native CUDA restore
Preserve optimizer, densification statistics, SH degree, RNG and sampler. Changes to build hashes and resource/logging fields are recorded in migration.json. No Gaussian reinitialization. Previous output is optional and must match the original input.

In [ ]:
extra = ['--previous',PREVIOUS] if PREVIOUS else []
stage('prepare',extra)
active = json.loads((WORK/'kaggle_active_run.json').read_text())
RUN = WORK/active['run']
print((RUN/'cuda_restore.json').read_text())
print((RUN/'migration.json').read_text())


## 6. Resume to the configured absolute session target
Total schedule stays30000; learning rates, density schedule, resolution and split stay unchanged. This cell starts the long GPU job only when you run it.

In [ ]:
try:
    stage('train',['--until',SESSION_TARGET])
finally:
    # On failure/interruption, preserve the latest fully published checkpoint if possible.
    stage('export',['--output',OUTPUT])


## 7. Checkpoint persistence
The run keeps the latest two full checkpoints, every1000 iterations plus the selected session target. Older checkpoints are removed only inside this new Kaggle workspace. Source input is untouched. Save Version with outputs after this session; /kaggle/working alone is not durable across every reset. Attach both original input and saved output to resume a later session. Abrupt machine termination may prevent the export cell from running; periodic checkpoints remain in the live workspace only until explicitly saved.

In [ ]:
print((RUN/'checkpoints/latest.json').read_text())
print('Saved handoff bundle:',OUTPUT)


## 8. Held-out evaluation after30000 only

In [ ]:
pointer=json.loads((RUN/'checkpoints/latest.json').read_text())
if pointer['iteration']==30000:
    stage('evaluate')
    print((RUN/'metrics/test_030000.json').read_text())
else:
    print('Partial session; final metrics unavailable until completed training')


## 9. Export final evaluated bundle
The first export preserves recovery state before evaluation. Export the evaluated version separately; select phase02_results_final for local final import. Download the bundle directory from the saved version Output tab, or use kaggle kernels output OWNER/SLUG -p LOCAL_DIRECTORY. Keep the setup log as evidence too.

In [ ]:
if pointer['iteration']==30000:
    FINAL=OUTPUT.with_name(OUTPUT.name+'_final')
    stage('export',['--output',FINAL])
    print('Download final bundle:',FINAL)
else:
    print('Download partial bundle:',OUTPUT)
